# 正则表达式

学习目标：能用正则完成校验、提取和替换，并正确处理 Unicode、匹配状态与动态文本转义。

前置知识：字符串、数组、对象、循环、函数与异常处理。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/16-regular-expressions/。

1. [patterns.mjs](scripts/16-regular-expressions/patterns.mjs)：正则创建、字符类与量词。
2. [groups-and-assertions.mjs](scripts/16-regular-expressions/groups-and-assertions.mjs)：命名捕获、反向引用与零宽断言。
3. [flags-and-state.mjs](scripts/16-regular-expressions/flags-and-state.mjs)：全局、粘连与独立校验。
4. [string-operations.mjs](scripts/16-regular-expressions/string-operations.mjs)：match、matchAll、replace 与 split。
5. [match-all-error.mjs](scripts/16-regular-expressions/match-all-error.mjs)：matchAll 缺少 g 标志的独立反例。
6. [unicode-and-indices.mjs](scripts/16-regular-expressions/unicode-and-indices.mjs)：Unicode 属性、v 集合和匹配索引。
7. [escape.mjs](scripts/16-regular-expressions/escape.mjs)：安全插入动态字面文本。
8. [backtracking.mjs](scripts/16-regular-expressions/backtracking.mjs)：短输入观察与消除嵌套重复。

## 1 模式、字符类和量词

正则表达式描述字符串匹配模式；RegExp 对象既可由字面量创建，也可由构造函数创建。固定模式用字面量较直观，动态模式用构造函数；构造参数若是普通字符串字面量，先经过字符串转义，再交给正则解析，因此反斜杠需要双写。

字符类 [A-Z] 表示一个 ASCII 大写字母，[^A-Z] 表示一个不在该集合内的字符；\d 表示 ASCII 数字，\s 表示空白，\w 主要表示 ASCII 字母、数字和下划线，不能当作任意语言的词。点号默认不匹配行终止符。量词作用于紧邻的前一项；以下 x 表示待重复的单个模式，n、m 是非负整数且 n ≤ m。

| 写法 | 中文名称／含义 |
| --- | --- |
| x* | 零次或多次 |
| x+ | 一次或多次 |
| x? | 零次或一次 |
| x{n} | 恰好 n 次 |
| x{n,} | 至少 n 次 |
| x{n,m} | n 到 m 次 |

量词默认贪婪，在允许成功匹配的条件下优先尝试更多重复；后接 ? 改为惰性，优先尝试较少重复。校验完整输入时应写清边界，而不只是判断里面是否出现过一段合格文本。

[patterns.mjs](scripts/16-regular-expressions/patterns.mjs)：

```javascript
const literal = /^[A-Z]{2}-\d{3}$/;
const constructed = new RegExp("^[A-Z]{2}-\\d{3}$");
console.log(literal.test("JS-012"), literal.test("JS-12"), constructed.test("JS-012"));
console.log(/^colou?r$/.test("color"), /^ab*c$/.test("ac"), /^ab+c$/.test("ac"));
console.log(/a{2,}/.exec("baaa")[0]);
console.log(/<.*>/.exec("<a><b>")[0], /<.*?>/.exec("<a><b>")[0]);
console.log(/[^0-9]+/.exec("12abc34")[0], /\s/.test(" "));

// 按本例输入运行，输出依次为：
// true false true
// true true false
// aaa
// <a><b> <a>
// abc true
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/patterns.mjs
```

## 2 捕获组、反向引用和断言

圆括号把模式分组；普通捕获组用序号取出匹配内容，(?&lt;name&gt;...) 用名称取出，(?:...) 仅分组而不捕获。这里 name 是组名，省略号代表该组内的具体模式，不是要匹配的文本。反向引用匹配先前组实际捕获的文本，数字写法如 \1，命名写法如 \k&lt;word&gt;。

断言检查位置而不消费字符：^ 与 $ 表示输入边界，m 标志使其也识别行边界；\b 以正则“词字符”的定义判断边界，不等于中文分词。(?=...) 与 (?!...) 分别检查后面匹配或不匹配，(?&lt;=...) 与 (?&lt;!...) 分别检查前面匹配或不匹配。exec 成功返回匹配数组及 groups、index 等信息，失败返回 null；下面给定能匹配的输入，直接读取捕获结果。需要根据“匹配/未匹配”采取不同操作时，才增加相应分支。

[groups-and-assertions.mjs](scripts/16-regular-expressions/groups-and-assertions.mjs)：

```javascript
const pair = /^(?<word>[a-z]+)-\k<word>$/;
const matched = pair.exec("go-go");
console.log(matched[0], matched.groups.word, matched.index);
console.log(pair.test("go-stop"), /^(ab)-\1$/.test("ab-ab"));
console.log(/(?:cat|dog)s?/.exec("dogs")[0]);
console.log(/\d+(?=kg)/.exec("12kg")[0]);
console.log(/(?<=USD )\d+/.exec("USD 25")[0]);
console.log(/a(?!b)/.test("ac"), /(?<!x)a/.test("ba"));
console.log(/\bcat\b/.test("a cat!"), /^b$/m.test("a\nb\nc"));

// 按本例输入运行，输出依次为：
// go-go go 0
// false true
// dogs
// 12
// 25
// true true
// true true
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/groups-and-assertions.mjs
```

## 3 标志与 lastIndex 状态

标志跟在字面量结束的斜杠后，或作为 RegExp 构造函数的第二参数。每个标志最多一次，u 与 v 不能同时使用。下表只列本章使用的标准标志。

| 标志原文名 | 中文名称／含义 |
| --- | --- |
| g | 全局匹配，后续匹配从 lastIndex 开始 |
| i | 忽略大小写 |
| m | 让输入边界断言也识别行边界 |
| s | 让点号也匹配行终止符 |
| u | 按 Unicode 码点处理及启用相应语法约束 |
| v | Unicode 集合模式，扩展 u 的能力 |
| y | 粘连匹配，必须恰从 lastIndex 处匹配 |
| d | 为匹配结果提供索引范围 |

g 或 y 的 test/exec 成功后更新 lastIndex，失败后重置为 0；g 可向后搜索，y 不跳过不匹配位置。共享带状态的正则做多次独立校验容易受之前调用影响，独立校验通常不用 g/y。exec 若匹配空字符串，可能不推进 lastIndex，手写循环要额外处理，或使用能处理空匹配推进的 matchAll。

[flags-and-state.mjs](scripts/16-regular-expressions/flags-and-state.mjs)：

```javascript
const global = /a/g;
console.log(global.test("a"), global.lastIndex);
console.log(global.test("a"), global.lastIndex);
console.log(/a/.test("a"), /a/.test("a"));
const sticky = /a/y;
console.log(sticky.test("ba"), sticky.lastIndex);
sticky.lastIndex = 1;
console.log(sticky.test("ba"), sticky.lastIndex);
console.log(/^js$/i.test("JS"), /a.b/s.test("a\nb"));
const empty = /(?:)/g;
console.log(empty.exec("a")[0].length, empty.lastIndex);
console.log([..."a".matchAll(/(?:)/g)].length);

// 按本例输入运行，输出依次为：
// true 1
// false 0
// true true
// false 0
// true 2
// true true
// 0 0
// 2
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/flags-and-state.mjs
```

## 4 批量提取、替换和切分

test 适合布尔判断，exec 适合取得一次匹配的详细结果。字符串 match 在没有 g 时返回含捕获信息的单次结果，有 g 时返回匹配文本数组，丢掉逐次捕获细节。matchAll 返回可遍历的逐次结果，传入正则时必须带 g；默认内置行为会复制匹配器状态而不改变原正则的 lastIndex，起点仍受传入正则当前 lastIndex 影响。

replace 返回新字符串，正则有 g 才替换所有匹配。替换模板可用 $1 或 $&lt;name&gt; 引用捕获组；回调则可计算替换值。split 用分隔模式切分，分隔模式含捕获组时组内容也进入结果，所以仅分组应写成非捕获组。

[string-operations.mjs](scripts/16-regular-expressions/string-operations.mjs)：

```javascript
const text = "JS:12 TS:8";
const pattern = /(?<name>[A-Z]+):(?<hours>\d+)/g;
console.log(text.match(/[A-Z]+/g).join(","));
const rows = [];
for (const match of text.matchAll(pattern)) {
  rows.push(match.groups.name + "=" + Number(match.groups.hours));
}
console.log(rows.join(","), pattern.lastIndex);
console.log(text.replace(/(?<name>[A-Z]+):(\d+)/g, "$<name>[$2]"));
console.log(text.replace(/\d+/g, digits => String(Number(digits) * 2)));
console.log(JSON.stringify("a, b; c".split(/[,;]\s*/)));
console.log(JSON.stringify("a,b".split(/(,)/)));

// 按本例输入运行，输出依次为：
// JS,TS
// JS=12,TS=8 0
// JS[12] TS[8]
// JS:24 TS:16
// ["a","b","c"]
// ["a",",","b"]
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/string-operations.mjs
```

[match-all-error.mjs](scripts/16-regular-expressions/match-all-error.mjs)：

```javascript
"JS TS".matchAll(/[A-Z]+/);

// 独立运行：退出状态为 1；诊断包含 TypeError；non-global RegExp。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/16-regular-expressions/match-all-error.mjs
```

## 5 Unicode 属性、v 集合与 d 索引

Unicode 属性转义在 u 或 v 模式中使用，例如 \p{Letter} 匹配字母，\P{Letter} 匹配其补集。它比 ASCII 范围更适合跨文字系统输入，但仍不自动处理字符串规范化。

v 已纳入 ECMAScript 2024，本课程 2025 基线包含它；字符类中可用 && 取交集、-- 取差集，还可用 \q{...} 表达有限字符串成员。字符类语法比旧模式严格，不能直接给旧模式加 v 就假设含义不变。d 已纳入 ECMAScript 2022，它提供 [开始, 结束) 区间；结束位置不包含在匹配内，索引仍按 UTF-16 码元计数，即使同时使用 u/v。本节 v/d 示例均可在 Node.js 24.11.0 默认模式运行。

[unicode-and-indices.mjs](scripts/16-regular-expressions/unicode-and-indices.mjs)：

```javascript
console.log(/^\p{Letter}+$/u.test("中文é"), /^\p{Letter}+$/u.test("JS1"));
const asciiLetters = /[\p{ASCII}&&\p{Letter}]+/v;
console.log(asciiLetters.exec("éABC中文")[0]);
console.log(/[[a-z]--[aeiou]]+/v.exec("ae-bcdf")[0]);
console.log(/^[\q{ab|cd}]$/v.test("ab"), /^[\q{ab|cd}]$/v.test("a"));
const match = /(?<face>😀)/du.exec("A😀B");
console.log(JSON.stringify(match.indices[0]), JSON.stringify(match.indices.groups.face));
const [start, end] = match.indices.groups.face;
console.log("A😀B".slice(start, end));

// 按本例输入运行，输出依次为：
// true false
// ABC
// bcdf
// true false
// [1,3] [1,3]
// 😀
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/unicode-and-indices.mjs
```

## 6 RegExp.escape 与动态文本

RegExp.escape 已纳入 ECMAScript 2025，在 Node.js 24.11.0 可直接调用。它把普通字符串转换成可嵌入模式的字面匹配片段，处理正则特殊字符、某些标点、空白，以及开头 ASCII 字母或数字与前一个转义相接的歧义；不要用只给点号加反斜杠的替换来冒充完整算法。

它要求输入是字符串，返回的是模式片段而非带斜杠的 RegExp 字面量。对本例固定的字符串边界校验，用转义结果拼接模式；这只能控制被插入文本的含义，不能修复外围模式设计不当产生的回溯问题。

[escape.mjs](scripts/16-regular-expressions/escape.mjs)：

```javascript
const requested = "a+b.js";
const escaped = RegExp.escape(requested);
const exact = new RegExp("^" + escaped + "$", "u");
console.log(escaped);
console.log(exact.test("a+b.js"), exact.test("aaabXjs"));
console.log(new RegExp(RegExp.escape("(draft)")).test("(draft)"));

// 按本例输入运行，输出依次为：
// \x61\+b\.js
// true false
// true
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/escape.mjs
```

## 7 过度回溯与输入边界

回溯引擎在某条尝试失败后会返回之前的选择点尝试另一种划分。嵌套重复且边界不明确时，相同输入可能被划分出大量路径；末尾失败会放大成本。ECMA-262 不承诺所有正则在线性时间内完成，也不能把某个引擎的实验性回退能力当成跨宿主保证。

下面的危险模式仅在短字符串上演示与较简单模式相同的布尔结果；这不是性能跑分。末段单独展示“长度至多 64 且只含指定字符”的匹配条件，长度比较是该条件本身，不是给其他示例添加的入口保护。应用中应简化重复结构、限定输入长度，复杂嵌套格式可使用解析器。把贪婪改为惰性并不普遍消除多条回溯路径。

[backtracking.mjs](scripts/16-regular-expressions/backtracking.mjs)：

```javascript
const ambiguous = /^(a+)+$/;
const simpler = /^a+$/;
for (const input of ["aaa", "aaaa!"]) {
  console.log(input, ambiguous.test(input), simpler.test(input));
}
function acceptsSmallToken(text) {
  return text.length <= 64 && /^[a-z0-9-]+$/u.test(text);
}
console.log(acceptsSmallToken("note-12"), acceptsSmallToken("a".repeat(65)));

// 按本例输入运行，输出依次为：
// aaa true true
// aaaa! false false
// true false
```

Step 1：运行本节示例。

```bash
node scripts/16-regular-expressions/backtracking.mjs
```

## 本章小结

- 模式负责匹配条件，test/exec 和字符串方法决定结果形式。
- g/y 会改变 lastIndex，Unicode 模式与 UTF-16 索引单位须区分。
- RegExp.escape 处理动态字面文本，回溯风险仍需从外围模式和输入边界控制。

## 练习

1. 匹配两位大写字母、连字符、四位数字，并用命名组取出编号。可核对标准：JS-0012 成功，js-0012 与 JS-012 失败，编号保留前导零。
2. 用 matchAll 取出输入 A1 B22 中的字母和数字。可核对标准：得到两组对应值，原正则 lastIndex 不变。
3. 用 RegExp.escape 构建匹配用户文本 [a+b] 的模式。可核对标准：它只匹配字面文本，而不把方括号或加号解释成模式控制字符。

### 提示

1. 把数字量词改为 4，捕获编号时不转成 Number。
2. 使用带 g 的新正则，其 lastIndex 初值为 0。
3. 先转义，再决定是查找子串还是完整文本匹配。


### 参考解析

1. `/^[A-Z]{2}-(?<id>\d{4})$/` 对 JS-0012 得到 groups.id 为字符串“0012”；其他两项 test 均为 false。
2. `/([A-Z])(\d+)/g` 的两次 matchAll 结果中，第 1、2 捕获依次是 A 与 1、B 与 22；原正则 lastIndex 仍为 0。
3. `new RegExp("^" + RegExp.escape("[a+b]") + "$", "u")` 对“[a+b]”为 true，对“ab”或“a+b”为 false。若去掉边界，含这段字面文本的更长字符串也能命中，应按任务明确匹配范围。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39 官方 ECMAScript 2025 分页版 | [§22.2.1–22.2.2 字符类、量词、捕获、断言与 Unicode 集合](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-patterns)；[§22.2.7.2 lastIndex、索引与匹配行为](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-regexpbuiltinexec)；[§22.2.5.1 RegExp.escape](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-regexp.escape)；[§22.1.3.13–14 match/matchAll](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-string.prototype.matchall)；[§22.1.3.19 replace 与 §22.1.3.23 split](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-string.prototype.replace)。 |
| V8 官方说明 | [v 标志的集合与字符串属性](https://v8.dev/features/regexp-v-flag)；[Background: catastrophic backtracking；引擎特性不等于标准保证](https://v8.dev/blog/non-backtracking-regexp)。 |
| MDN 用法对照 | [创建、转义与常用方法](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Regular_expressions)；[RegExp.escape 的字符串输入与转义边界](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/RegExp/escape)。 |
